# Applied Programming for AI
## Mini-project 1 — Service Ticket Data Pipeline

**Learning outcomes**
* Turn data quality rules into a reusable NumPy/Pandas workflow.
* Preserve rejected data, transformation flags and an audit trail.
* Check edge cases and export reproducible results.

**Summary:** 1. Diagnosis · 2. Rejection policy · 3. Category normalization · 4. Numeric treatment · 5. Features · 6. Reusable function · 7. Edge cases · 8. Export and report

Read [BRIEF.md](https://github.com/tgvp/Programming-AI/blob/main/prj/Project-1/BRIEF.md) before coding. Keep the synthetic input unchanged. Restart and run all cells from the beginning before submitting. Contribution: 1.0/20 (5%).

**Colab:** upload `service_tickets.csv` with the Files panel (left sidebar) before running the first code cell.

### Importing libraries and loading the data

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

candidates=[Path('service_tickets.csv'),Path('prj/Project-1/service_tickets.csv')]
source=next((p for p in candidates if p.is_file()),None)
if source is None: raise FileNotFoundError('Open from the project folder or upload service_tickets.csv to Colab')
raw=pd.read_csv(source)
print(raw.head())

  ticket_id   opened_on channel priority handling_minutes  satisfaction
0      T001  2026-10-01   email   medium             42.2           2.0
1      T002  2026-10-01   phone     high             76.3           3.0
2      T003  2026-10-01     web      low              NaN           4.0
3      T004  2026-10-01   email   medium            118.0           5.0
4      T005  2026-10-02   phone     high             99.5           1.0


## 1. Diagnosis
Report shape, schema, missingness, exact duplicates and repeated IDs. Explain the observation unit.

In [2]:
print('Shape (rows, columns):', raw.shape)
print('\nSchema:'); print(raw.dtypes)
print('\nMissing values per column (already missing in the source):'); print(raw.isna().sum())

n_exact = int(raw.duplicated().sum())
print('\nExact duplicate rows (extra copies):', n_exact)

deduped_view = raw.drop_duplicates()
ids = deduped_view['ticket_id'].astype('string')
well_formed = ids.str.fullmatch(r'T\d{3}').fillna(False).astype(bool)
print('Missing IDs:', int(ids.isna().sum()), '| present but malformed IDs:', int((ids.notna() & ~well_formed).sum()))
repeated = deduped_view[well_formed & ids.duplicated(keep=False)].sort_values('ticket_id')
print('Rows whose ID is repeated with different data (after removing exact copies):', len(repeated))
print(repeated.to_string())

# Present-but-invalid values (different from missing)
dates = pd.to_datetime(raw['opened_on'].astype('string').str.strip(), errors='coerce', format='mixed')
print('\nopened_on: missing =', int(raw['opened_on'].isna().sum()),
      '| present but unparseable =', int((raw['opened_on'].notna() & dates.isna()).sum()),
      '| parsed but outside 1-31 Oct 2026 =', int((dates.notna() & ~dates.dt.normalize().between('2026-10-01','2026-10-31')).sum()))
dur = pd.to_numeric(raw['handling_minutes'], errors='coerce')
print('handling_minutes: missing =', int(raw['handling_minutes'].isna().sum()),
      '| unparseable =', int((raw['handling_minutes'].notna() & dur.isna()).sum()),
      '| negative =', int((dur < 0).sum()), '| non-finite =', int(np.isinf(dur).sum()))
sat = pd.to_numeric(raw['satisfaction'], errors='coerce')
sat_ok = np.isfinite(sat) & (sat == np.floor(sat)) & sat.between(1, 5)
print('satisfaction: missing =', int(raw['satisfaction'].isna().sum()),
      '| present but invalid (unparseable, fractional or outside 1-5) =', int((raw['satisfaction'].notna() & ~sat_ok).sum()))
for c in ['channel', 'priority']:
    print(f'\n{c} (raw values):'); print(raw[c].value_counts(dropna=False))

Shape (rows, columns): (84, 6)

Schema:
ticket_id               str
opened_on               str
channel                 str
priority                str
handling_minutes        str
satisfaction        float64
dtype: object

Missing values per column (already missing in the source):
ticket_id           1
opened_on           0
channel             1
priority            1
handling_minutes    1
satisfaction        1
dtype: int64

Exact duplicate rows (extra copies): 3
Missing IDs: 1 | present but malformed IDs: 0
Rows whose ID is repeated with different data (after removing exact copies): 2
   ticket_id   opened_on channel priority handling_minutes  satisfaction
39      T040  2026-10-10   email   medium             52.8           1.0
80      T040  2026-10-10   email   medium              999           1.0

opened_on: missing = 0 | present but unparseable = 1 | parsed but outside 1-31 Oct 2026 = 0
handling_minutes: missing = 1 | unparseable = 1 | negative = 1 | non-finite = 0
satisfaction: mi

**Observation unit.** One row is one *ticket observation* (a ticket as recorded at one moment), not necessarily one unique ticket: the same `ticket_id` can appear several times. An exact copy is a harmless duplicate record; the same ID with *different* data is a conflict we cannot resolve without outside information. **Missing** means the source has no value; **invalid** means a value is present but breaks the field's rule (e.g. `-3` minutes, rating `3.5`, date `2026-11-03`). The two are handled differently below.

## 2. Rejection policy
Remove exact copies. Identify conflicting IDs and invalid IDs/dates. Preserve rejected source rows and all reasons.

In [3]:
ID_PATTERN = r'T\d{3}'
OCT_START, OCT_END = pd.Timestamp('2026-10-01'), pd.Timestamp('2026-10-31')

def drop_exact_duplicates(df):
    """Remove only the extra copies of fully identical rows (keep the first)."""
    dup = df.duplicated(keep='first')
    return df.loc[~dup].copy(), int(dup.sum())

def parse_dates(s):
    try:
        return pd.to_datetime(s, errors='coerce', format='mixed')
    except (ValueError, TypeError):
        return pd.to_datetime(s, errors='coerce')

def rejection_reasons(df):
    """Return (reasons, parsed_dates). reasons = ';'-joined text per row, '' if the row is kept."""
    ids = df['ticket_id'].astype('string')
    missing_id = (ids.isna() | (ids.str.strip() == '')).astype(bool)
    well_formed = ids.str.fullmatch(ID_PATTERN).fillna(False).astype(bool)
    malformed_id = (~missing_id & ~well_formed).astype(bool)
    # every occurrence of a well-formed ID that is still repeated (rows differ, since exact copies are gone)
    conflicting_id = (well_formed & ids.duplicated(keep=False)).astype(bool)

    raw_dates = df['opened_on'].astype('string').str.strip()
    missing_date = (raw_dates.isna() | (raw_dates == '')).astype(bool)
    parsed = parse_dates(raw_dates)
    unparseable_date = (~missing_date & parsed.isna()).astype(bool)
    out_of_october = (parsed.notna() & ~parsed.dt.normalize().between(OCT_START, OCT_END)).astype(bool)

    masks = pd.DataFrame({'missing_id': missing_id, 'malformed_id': malformed_id,
                          'conflicting_id': conflicting_id, 'missing_date': missing_date,
                          'unparseable_date': unparseable_date, 'out_of_october_2026': out_of_october},
                         index=df.index)
    reasons = pd.Series([';'.join(c for c in masks.columns if row[c]) for row in masks.to_dict('records')],
                        index=df.index, dtype='object')
    return reasons, parsed

deduped, n_dups = drop_exact_duplicates(raw)
reasons, parsed_dates = rejection_reasons(deduped)
rejected_mask = reasons != ''
print('Exact duplicates removed:', n_dups)
print('Rejected rows (each counted once):', int(rejected_mask.sum()))
print(reasons[rejected_mask].value_counts())

Exact duplicates removed: 3
Rejected rows (each counted once): 4
conflicting_id      2
unparseable_date    1
missing_id          1
Name: count, dtype: int64


## 3. Category normalization
Normalize channel and priority and preserve unknown flags according to the brief.

In [4]:
VALID_CHANNELS = {'email', 'phone', 'web'}
VALID_PRIORITIES = {'low', 'medium', 'high'}

def normalize_category(s, valid):
    """Strip + lowercase; missing/unrecognised -> 'unknown'. Returns (values, unknown_flag)."""
    t = s.astype('string').str.strip().str.lower()
    unknown = (t.isna() | ~t.isin(valid)).astype(bool)
    return t.where(~unknown, 'unknown').astype(object), unknown

kept_demo = deduped.loc[~rejected_mask]
ch, ch_unk = normalize_category(kept_demo['channel'], VALID_CHANNELS)
pr, pr_unk = normalize_category(kept_demo['priority'], VALID_PRIORITIES)
print(ch.value_counts()); print(pr.value_counts())
print('channel_unknown:', int(ch_unk.sum()), '| priority_unknown:', int(pr_unk.sum()))

channel
email      26
phone      25
web        24
unknown     2
Name: count, dtype: int64
priority
high       26
low        25
medium     24
unknown     2
Name: count, dtype: int64
channel_unknown: 2 | priority_unknown: 2


## 4. Numeric treatment
Treat durations and satisfaction differently. Preserve flags and explain why ratings must not be imputed.

In [5]:
def fix_durations(df):
    """Invalid/missing durations -> median of valid durations in the retained table. Returns (values, imputed_flag, fill)."""
    x = pd.to_numeric(df['handling_minutes'], errors='coerce').astype(float)
    valid = np.isfinite(x) & (x >= 0)          # zero and fractional values are valid
    if not valid.any():
        raise ValueError('No valid handling_minutes values remain in the retained table; '
                         'cannot compute a median fill value (no zero fallback is used).')
    fill = float(x[valid].median())
    return x.where(valid, fill), (~valid).astype(bool), fill

def fix_satisfaction(s):
    """Only integers 1-5 are valid. Everything else becomes missing (never imputed)."""
    x = pd.to_numeric(s, errors='coerce').astype(float)
    valid = np.isfinite(x) & (x == np.floor(x)) & x.between(1, 5)
    return x.where(valid).astype('Int64'), (~valid).astype(bool)

minutes, imputed, fill = fix_durations(kept_demo)
sat_vals, sat_unavail = fix_satisfaction(kept_demo['satisfaction'])
print('Duration fill value (median of valid):', fill, '| imputed:', int(imputed.sum()))
print('Satisfaction unavailable:', int(sat_unavail.sum()))

Duration fill value (median of valid): 59.0 | imputed: 3
Satisfaction unavailable: 3


**Why durations are imputed but ratings are not.** A missing duration is a *measurement gap* about a quantity that is always defined (every ticket took some time), so a neutral value (the median) keeps the row usable, and `handling_imputed` records that it was filled. A satisfaction rating is a *customer opinion*: if the customer did not rate, no value exists. Filling it would invent opinions that nobody expressed and bias any later analysis, so it stays missing and is flagged with `satisfaction_unavailable`.

## 5. Features
Compute safe duration scaling and the four required channel indicators; verify identifier alignment.

In [6]:
def zscore(x):
    """Population z-score (ddof=0). Constant column -> all zeros (safe denominator of 1)."""
    x = np.asarray(x, dtype=float)
    if x.size == 0 or x.max() == x.min():
        return np.zeros_like(x)
    return (x - x.mean()) / x.std(ddof=0)

CHANNEL_COLS = ['channel_email', 'channel_phone', 'channel_web', 'channel_unknown']

def build_features(clean):
    feats = pd.DataFrame({'ticket_id': clean['ticket_id'].to_numpy(),
                          'handling_zscore': clean['handling_zscore'].to_numpy()})
    for name in ['email', 'phone', 'web', 'unknown']:
        feats[f'channel_{name}'] = (clean['channel'].to_numpy() == name).astype(int)
    return feats   # satisfaction is deliberately NOT a feature

z = zscore(minutes.to_numpy())
print('mean ~', round(float(z.mean()), 12), '| std ~', round(float(z.std(ddof=0)), 12))

mean ~ 0.0 | std ~ 1.0


## 6. Reusable function
Consolidate your work into `clean_tickets(raw)`, returning clean, rejected, features and audit. The input is not mutated.

In [7]:
def clean_tickets(raw):
    df = raw.copy()                                   # never touch the caller's table
    raw_count = len(df)

    deduped, n_dups = drop_exact_duplicates(df)
    reasons, parsed = rejection_reasons(deduped)
    rej_mask = reasons != ''

    rejected = deduped.loc[rej_mask].copy()           # original source values preserved
    rejected.insert(0, 'source_row', rejected.index)  # row position in the raw table
    rejected['rejection_reason'] = reasons[rej_mask]
    rejected = rejected.reset_index(drop=True)

    kept = deduped.loc[~rej_mask]
    kept_dates = parsed.loc[~rej_mask]

    channel, channel_unknown = normalize_category(kept['channel'], VALID_CHANNELS)
    priority, priority_unknown = normalize_category(kept['priority'], VALID_PRIORITIES)
    minutes, handling_imputed, fill = fix_durations(kept)   # raises a clear error if nothing valid
    satisfaction, satisfaction_unavailable = fix_satisfaction(kept['satisfaction'])

    clean = pd.DataFrame({
        'ticket_id': kept['ticket_id'].astype(str),
        'opened_on': kept_dates.dt.strftime('%Y-%m-%d'),
        'channel': channel, 'priority': priority,
        'handling_minutes': minutes, 'satisfaction': satisfaction,
        'channel_unknown': channel_unknown, 'priority_unknown': priority_unknown,
        'handling_imputed': handling_imputed, 'satisfaction_unavailable': satisfaction_unavailable,
        'handling_zscore': zscore(minutes.to_numpy()),
    }, index=kept.index).reset_index(drop=True)

    features = build_features(clean)

    assert raw_count == n_dups + len(rejected) + len(clean), 'row balance broken'
    reason_counts = {}
    for r in rejected['rejection_reason']:
        for part in r.split(';'):
            reason_counts[part] = reason_counts.get(part, 0) + 1
    audit = {
        'raw_count': int(raw_count),
        'exact_duplicates_removed': int(n_dups),
        'rejected_count': int(len(rejected)),
        'retained_count': int(len(clean)),
        'unknown_channel_count': int(clean['channel_unknown'].sum()),
        'unknown_priority_count': int(clean['priority_unknown'].sum()),
        'imputed_duration_count': int(clean['handling_imputed'].sum()),
        'unavailable_satisfaction_count': int(clean['satisfaction_unavailable'].sum()),
        'duration_fill_value': float(fill),
        'duration_mean': float(minutes.mean()),
        'duration_std_population': float(minutes.std(ddof=0)),
        'rejection_reason_counts': reason_counts,
    }
    return clean, rejected, features, audit

snapshot = raw.copy(deep=True)
clean, rejected, features, audit = clean_tickets(raw)
pd.testing.assert_frame_equal(raw, snapshot)          # input not mutated
print(json.dumps(audit, indent=2))
print(clean.head())

{
  "raw_count": 84,
  "exact_duplicates_removed": 3,
  "rejected_count": 4,
  "retained_count": 77,
  "unknown_channel_count": 2,
  "unknown_priority_count": 2,
  "imputed_duration_count": 3,
  "unavailable_satisfaction_count": 3,
  "duration_fill_value": 59.0,
  "duration_mean": 59.589610389610385,
  "duration_std_population": 30.595274163617248,
  "rejection_reason_counts": {
    "unparseable_date": 1,
    "missing_id": 1,
    "conflicting_id": 2
  }
}
  ticket_id   opened_on channel priority  handling_minutes  satisfaction  \
0      T001  2026-10-01   email   medium              42.2             2   
1      T002  2026-10-01   phone     high              76.3             3   
2      T003  2026-10-01     web      low              59.0             4   
3      T004  2026-10-01   email   medium             118.0             5   
4      T005  2026-10-02   phone     high              99.5             1   

   channel_unknown  priority_unknown  handling_imputed  \
0            False       

## 7. Edge cases
Test the two-ticket constant-duration example, conflicting IDs, dates, multiple reasons, valid zero/fractional durations, fractional ratings and the no-valid-duration error.

In [8]:
COLS = ['ticket_id', 'opened_on', 'channel', 'priority', 'handling_minutes', 'satisfaction']

# (a) Worked case from the brief
two = pd.DataFrame([['T001', '2026-10-01', 'email', 'low', 10, 5],
                    ['T002', '2026-10-02', 'web', 'high', None, None]], columns=COLS)
c, r, f, a = clean_tickets(two)
assert c['handling_minutes'].tolist() == [10.0, 10.0]
assert c['handling_imputed'].tolist() == [False, True]
assert c['handling_zscore'].tolist() == [0.0, 0.0]
assert len(r) == 0 and len(c) == 2
print('(a) two-ticket example OK')

# (b) Mixed edge cases
rows = [
 ['T001', '2026-10-05', 'email', 'low', 10, 5],        # kept
 ['T001', '2026-10-05', 'email', 'low', 10, 5],        # exact duplicate -> only the extra copy removed
 ['T002', '2026-10-06', 'Web ', 'HIGH', 20, 3],        # conflicting ID (both rows rejected)
 ['T002', '2026-10-06', 'web', 'high', 25, 3],
 ['T003', None, 'phone', 'medium', 5, 4],              # missing date
 ['T004', 'garbage', 'phone', 'medium', 5, 4],         # unparseable date
 ['T005', '2026-09-30', 'phone', 'medium', 5, 4],      # out of October
 ['X9',   '2026-11-05', 'phone', 'medium', 5, 4],      # malformed ID + out of October (2 reasons, 1 row)
 [None,   '2026-10-07', 'phone', 'medium', 5, 4],      # missing ID
 ['T006', '2026-10-31', 'fax', None, 0, 3.5],          # zero minutes valid; unknown cat; fractional rating -> missing
 ['T007', '2026-10-01', 'email', 'medium', 2.5, 6],    # fractional minutes valid; rating 6 -> missing
 ['T008', '2026-10-10', 'web', 'low', -3, 'abc'],      # negative minutes -> imputed
 ['T009', '2026-10-11', 'phone', 'low', 'abc', 2],     # unparseable minutes -> imputed
]
t = pd.DataFrame(rows, columns=COLS)
before = t.copy(deep=True)
c, r, f, a = clean_tickets(t)
pd.testing.assert_frame_equal(t, before)
assert a['raw_count'] == a['exact_duplicates_removed'] + a['rejected_count'] + a['retained_count'] == 13
assert a['exact_duplicates_removed'] == 1 and a['rejected_count'] == 7 and a['retained_count'] == 5
assert (c['ticket_id'] == 'T001').sum() == 1
assert set(r.loc[r['ticket_id'] == 'T002', 'rejection_reason']) == {'conflicting_id'} and (r['ticket_id'] == 'T002').sum() == 2
assert r.loc[r['ticket_id'] == 'X9', 'rejection_reason'].iloc[0] == 'malformed_id;out_of_october_2026'
assert a['duration_fill_value'] == 2.5 and a['imputed_duration_count'] == 2
assert c.loc[c['ticket_id'] == 'T006', 'handling_minutes'].iloc[0] == 0 and not c.loc[c['ticket_id'] == 'T006', 'handling_imputed'].iloc[0]
assert c.loc[c['ticket_id'] == 'T007', 'handling_minutes'].iloc[0] == 2.5
assert c['satisfaction'].isna().sum() == 3 and a['unavailable_satisfaction_count'] == 3
assert a['unknown_channel_count'] == 1 and a['unknown_priority_count'] == 1
assert (f[CHANNEL_COLS].sum(axis=1) == 1).all() and f['ticket_id'].tolist() == c['ticket_id'].tolist()
print('(b) mixed edge cases OK')
print(r[['source_row', 'ticket_id', 'rejection_reason']])

# (c) No valid duration -> clear error (no zero fallback)
bad = pd.DataFrame([['T001', '2026-10-01', 'email', 'low', None, 5],
                    ['T002', '2026-10-02', 'web', 'low', -1, 4]], columns=COLS)
try:
    clean_tickets(bad)
    raise AssertionError('expected ValueError')
except ValueError as e:
    print('(c) error raised as expected:', e)

(a) two-ticket example OK
(b) mixed edge cases OK
   source_row ticket_id                  rejection_reason
0           2      T002                    conflicting_id
1           3      T002                    conflicting_id
2           4      T003                      missing_date
3           5      T004                  unparseable_date
4           6      T005               out_of_october_2026
5           7        X9  malformed_id;out_of_october_2026
6           8       NaN                        missing_id
(c) error raised as expected: No valid handling_minutes values remain in the retained table; cannot compute a median fill value (no zero fallback is used).


## 8. Export and report
Write required files to `outputs/`. Read the CSV files back, check row balance, and prepare `REPORT.md`.

In [9]:
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
clean.to_csv(OUT / 'clean_tickets.csv', index=False)
rejected.to_csv(OUT / 'rejected_tickets.csv', index=False)
features.to_csv(OUT / 'ticket_features.csv', index=False)
(OUT / 'audit.json').write_text(json.dumps(audit, indent=2))

# Read everything back
clean_rb = pd.read_csv(OUT / 'clean_tickets.csv')
rejected_rb = pd.read_csv(OUT / 'rejected_tickets.csv')
features_rb = pd.read_csv(OUT / 'ticket_features.csv')
audit_rb = json.loads((OUT / 'audit.json').read_text())

# Acceptance checks on the files as read back
assert audit_rb['raw_count'] == audit_rb['exact_duplicates_removed'] + len(rejected_rb) + len(clean_rb)
assert len(rejected_rb) == audit_rb['rejected_count'] and len(clean_rb) == audit_rb['retained_count']
assert clean_rb['ticket_id'].notna().all() and clean_rb['ticket_id'].str.fullmatch(r'T\d{3}').all()
assert clean_rb['ticket_id'].is_unique
d = pd.to_datetime(clean_rb['opened_on'])
assert d.between('2026-10-01', '2026-10-31').all()
assert np.isfinite(clean_rb['handling_minutes']).all() and (clean_rb['handling_minutes'] >= 0).all()
assert clean_rb['channel'].isin(['email', 'phone', 'web', 'unknown']).all()
assert clean_rb['priority'].isin(['low', 'medium', 'high', 'unknown']).all()
s = clean_rb['satisfaction'].dropna()
assert ((s == np.floor(s)) & s.between(1, 5)).all()
assert features_rb['ticket_id'].tolist() == clean_rb['ticket_id'].tolist()
assert (features_rb[CHANNEL_COLS].sum(axis=1) == 1).all()
zs = features_rb['handling_zscore']
if clean_rb['handling_minutes'].nunique() > 1:
    assert abs(zs.mean()) < 1e-9 and abs(zs.std(ddof=0) - 1) < 1e-9
else:
    assert (zs == 0).all()
assert rejected_rb['rejection_reason'].notna().all()
print('All acceptance checks passed.')
print(json.dumps(audit_rb, indent=2))

All acceptance checks passed.
{
  "raw_count": 84,
  "exact_duplicates_removed": 3,
  "rejected_count": 4,
  "retained_count": 77,
  "unknown_channel_count": 2,
  "unknown_priority_count": 2,
  "imputed_duration_count": 3,
  "unavailable_satisfaction_count": 3,
  "duration_fill_value": 59.0,
  "duration_mean": 59.589610389610385,
  "duration_std_population": 30.595274163617248,
  "rejection_reason_counts": {
    "unparseable_date": 1,
    "missing_id": 1,
    "conflicting_id": 2
  }
}


In [10]:
a = audit_rb
reasons_txt = ', '.join(f'{k}: {v}' for k, v in sorted(a['rejection_reason_counts'].items())) or 'none'
report = f"""# Mini-project 1 — Service Ticket Data Pipeline

## Diagnosis
Each row of `service_tickets.csv` is a ticket *observation*, so one `ticket_id` may appear several times. The raw file has {a['raw_count']} rows. I distinguished values that are missing in the source from values that are present but invalid (e.g. negative minutes, fractional ratings, dates outside October 2026).

## Rules
1. The raw CSV is never modified; the pipeline works on a copy.
2. Only exact duplicate rows are removed ({a['exact_duplicates_removed']} extra copies). Every occurrence of an ID that is still repeated with conflicting data is rejected, as are rows with a missing or malformed ID (not `T` + 3 digits) or a missing, unparseable or out-of-October-2026 date. All reasons are kept in `rejection_reason`; each row is counted once. Rejected rows: {a['rejected_count']} ({reasons_txt}).
3. `channel` and `priority` are stripped and lowercased; missing or unrecognised values become `unknown` with flags (unknown channels: {a['unknown_channel_count']}, unknown priorities: {a['unknown_priority_count']}).
4. Invalid or missing `handling_minutes` are filled with the median of valid retained durations ({a['duration_fill_value']}) and flagged: {a['imputed_duration_count']} rows. Zero and fractional values are valid. If no valid duration exists, the code raises an error.
5. `satisfaction` is valid only as an integer 1-5; otherwise it becomes missing and is flagged (`satisfaction_unavailable`: {a['unavailable_satisfaction_count']}). It is never imputed, because that would invent customer opinions.
6. `handling_zscore` uses mean {a['duration_mean']:.3f} and population standard deviation {a['duration_std_population']:.3f} (ddof=0; constant column gives zeros). The feature table holds the ID, the z-score and four one-hot channel columns; the rating is not a feature.

## Checks
Row balance: {a['raw_count']} = {a['exact_duplicates_removed']} duplicates + {a['rejected_count']} rejected + {a['retained_count']} retained. Retained IDs are well formed and unique, dates fall in October 2026, durations are finite and nonnegative, categories are known or `unknown`, ratings are integers 1-5 or missing, feature IDs match the clean table with one channel indicator per row, and all files were read back successfully. Edge cases (conflicting IDs, multiple reasons, constant duration, no valid duration) are tested in section 7.

## Limitations
- Rejecting every row of a conflicting ID removes tickets that may be valid, which shrinks the sample and may bias it.
- Median imputation shrinks the variance of durations, and missing ratings may not be random, so observed satisfaction may not represent all tickets. The data are also synthetic.

## Run instructions
Open `project_1_student.ipynb` in Google Colab, upload `service_tickets.csv` in the Files panel, choose Runtime > Restart and run all. Outputs are written to `outputs/`. Locally: `pip install numpy pandas jupyter`, then run the notebook from the repository root.

## Assistance used
Describe the assistance you used here (e.g. an AI assistant helped draft the code structure; I reviewed, ran and can explain every step).
"""
Path('REPORT.md').write_text(report)
print(len(report.split()), 'words')

450 words
